In [30]:
from datasets import load_dataset
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score
import numpy as np

ds = load_dataset("CMU-OSPO/protostar-climate-train-to-2018", "annual_indices")
df = ds["train"].to_pandas()

target = "Avg Temp"
df = df.sort_values(["ID", "Year"])

for k in (1, 2, 3):
    df[f"lag_{k}"] = df.groupby("ID")[target].shift(k)

In [31]:
features = df.select_dtypes(include="number").columns.tolist()
features.remove(target)

data = df.replace([np.inf, -np.inf], np.nan)
data = data.dropna(subset=features + [target])

In [32]:
cutoff_year = data["Year"].quantile(0.8)

train = data[data["Year"] <= cutoff_year]
test = data[data["Year"] > cutoff_year]

model = RandomForestRegressor(
    n_estimators=100,
    random_state=42
)

In [33]:
model.fit(train[features], train[target])

pred = model.predict(test[features])

print("features:", features)
print("r2:", r2_score(test[target], pred))

features: ['Lat', 'Lon', 'Year', 'Miss Tmax', 'Miss Tmin', 'Miss Prcp', 'Avg Tmax', 'Avg Tmin', 'ToT Prcp', 'Max Tmax', 'Min Tmax', 'Max Tmin', 'Min Tmin', 'Warm Days', 'Warm Nights', 'Cold Days', 'Cold Nights', 'Max 1-day P', 'Max 5-day P', 'Wet Days', 'cdd', 'lag_1', 'lag_2', 'lag_3']
r2: 0.9997221654071085
